# Домашнее задание 3

**Дисциплина** Алгоритмы и структуры данных

**Тема** Алгоритмы на строках

# Задание 1. КМП и нечеткий поиск

| Ограничение | Значение |
| ----------- | -------- |
| Ограничение времени | 1 секунда |
| Ограничение памяти | 64 Мб |
| Ввод | input.txt |
| Вывод | output.txt |

Дан текст. Для каждого слова в тексте вычисляется префикс-функция (π-функция) как в алгоритме Кнута-Морриса-Пратта (КМП). Необходимо:
- найти слово с максимальной суммой элементов его π-функции;
- если таких слов несколько, выбрать самое длинное из них;
- если и таких слов несколько, выбрать первое в алфавитном порядке.

Для найденного слова определить количество слов в тексте, которые отличаются от него не более чем на k по расстоянию Левенштейна и при этом не совпадают с искомым словом.

Расстояние Левенштейна - минимальное количество операций вставки, удаления и замены символа, необходимых для превращения одной строки в другую.

При обработке текста регистр не учитывается (прописная буква формально равна строчной).

**Формат ввода**

На первой строчке дан текст. На второй - значение k - количество возможных операций трансформации слова (расстояние Левенштейна).

**Формат вывода**

В выводе должны быть три строки:
- на первой строке - найденное слово;
- на второй строке - сумма элементов π-функции;
- на третьей - количество слов, которые можно получить из искомого не более чем за k операций (само искомое слово не учитывается).

In [1]:
data = '''аа ababaac ababaaa ababaa
1
'''

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(data)

In [2]:
import re

def compute_prefix_function(s):
    n = len(s)
    pi = [0] * n
    for i in range(1, n):
        j = pi[i - 1]
        while j > 0 and s[i] != s[j]:
            j = pi[j - 1]
        if s[i] == s[j]:
            j += 1
        pi[i] = j
    return pi


def levenshtein_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],
                    dp[i][j - 1],
                    dp[i - 1][j - 1]
                )
    return dp[m][n]


def solve():
    with open('input.txt', 'r', encoding='utf-8') as f:
        text = f.readline().strip()
        k = int(f.readline().strip())

    words = text.split()
    words_lower = [re.sub(r'[^\w]', '', word.lower()) for word in words]
    words_lower = [word for word in words_lower if word]

    word_info = []
    seen = {}
    for idx, word in enumerate(words_lower):
        if word not in seen:
            pi_sum = sum(compute_prefix_function(word))
            word_info.append({
                'word': word,
                'pi_sum': pi_sum,
                'length': len(word),
                'first_occurrence': idx
            })
            seen[word] = True

    max_pi_sum = max(info['pi_sum'] for info in word_info)
    candidates = [info for info in word_info if info['pi_sum'] == max_pi_sum]
    max_length = max(c['length'] for c in candidates)
    candidates = [c for c in candidates if c['length'] == max_length]
    candidates.sort(key=lambda x: x['first_occurrence'])

    target_word = candidates[0]['word']
    target_pi_sum = candidates[0]['pi_sum']

    count = 0
    for word in words_lower:
        if word != target_word and levenshtein_distance(target_word, word) <= k:
            count += 1

    with open('output.txt', 'w', encoding='utf-8') as f:
        f.write(f"{target_word}\n")
        f.write(f"{target_pi_sum}\n")
        f.write(f"{count}\n")


solve()

# Задание 2. Кодирование с разбиением

| Ограничение | Значение |
| ----------- | -------- |
| Ограничение времени | 5 секунд |
| Ограничение памяти | 64 Мб |
| Ввод | input.txt |
| Вывод | output.txt |

Дана битовая строка, закодированная с помощью кода Хаффмана. Необходимо:
1. Декодировать строку, используя известные коды Хаффмана.
2. Разделить декодированную строку на максимально возможное количество неповторяющихся подстрок.
3. Определить количество получившихся подстрок.

**Формат ввода**

В первой строке указано целое число $n$ - количество символов в алфавите.

На следующих $n$ строках приведен кодовый алфавит в формате: (символ, код).

Последняя строка - это битовая строка, которую надо декодировать и разделить на подстроки.

**Формат вывода**

Вывести количество неповторяющихся подстрок, на которые можно разбить декодированную строку.

In [3]:
data = '''6
(e, 100)
(h, 000)
(i, 101)
(s, 01)
(t, 11)
(x, 001)
1100010101101011110000111
'''

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(data)

In [4]:
def decode_huffman(encoded_string, huffman_codes):
    reverse_codes = {code: char for char, code in huffman_codes.items()}
    decoded = []
    current_code = ""

    for bit in encoded_string:
        current_code += bit
        if current_code in reverse_codes:
            decoded.append(reverse_codes[current_code])
            current_code = ""

    return ''.join(decoded)


def max_unique_substrings(s):
    def backtrack(start, seen):
        if start == len(s):
            return 0

        max_count = 0
        for end in range(start + 1, len(s) + 1):
            substring = s[start:end]
            if substring not in seen:
                seen.add(substring)
                count = 1 + backtrack(end, seen)
                max_count = max(max_count, count)
                seen.remove(substring)

        return max_count

    return backtrack(0, set())


def solve():
    with open('input.txt', 'r', encoding='utf-8') as f:
        n = int(f.readline().strip())

        huffman_codes = {}
        for _ in range(n):
            line = f.readline().strip()
            line = line.strip('()')
            parts = line.split(', ')
            char = parts[0]
            code = parts[1]
            huffman_codes[char] = code

        encoded_string = f.readline().strip()

    decoded_string = decode_huffman(encoded_string, huffman_codes)
    result = max_unique_substrings(decoded_string)

    with open('output.txt', 'w', encoding='utf-8') as f:
        f.write(f"{result}\n")


solve()